# Обучатель-наблюдатель против сетей

`research_m5/cells.py` — один обучатель для любого семейства задач «прочитай последовательность — дай ответ».
Он ищет **маленькую машину**: конечное управление (номер состояния или несколько переключателей) и рядом
до двух чисел (счётчик, уровень с краями, круг). Машину находит решатель SAT; градиентов нет.

**Протокол наблюдателя** (README, этапы 72–74). Ответ виден после каждого символа — как табло автомата,
на котором нажимают кнопки. Учение: 300 последовательностей длиной 1–4 и не больше 40 вопросов; вопрос —
одна последовательность, сколько бы символов в ней ни было. Проверка: 300 последовательностей длиной 20,
100 и 500. Сетям-соперникам то же самое: каждое начало учебных рассказов — свой пример, плюс 40 рассказов
длиной 8–32 на каждые 300.

**Что делает обучатель** (этап 74):

- сначала пробует короткие формы в порядке длины описания в битах;
- потом машины, у которых **состояния — это ответы** (то, что видно), со скрытой ячейкой или несколькими
  скрытыми значениями для того, чего не видно; из объясняющих рассказы берётся та, которой нужно меньше всего
  сказать («иначе ничего не происходит» не говорится);
- **сам спрашивает**: решатель ищет вторую машину, которая тоже объясняет всё услышанное, вместе с рассказом,
  на котором они расходятся; рассказ продолжается обходом ещё не сделанных ходов машины;
- последняя проверка идёт на рассказах до 100 символов; если вопросы кончились раньше или вторая машина не
  исключена, ответ выдаётся с меткой **«не уверен»**.

**Что записано** на 72 открытых семействах (шесть пулов, каждый написан исполнителями, не видевшими
обучателя; 95 % и выше на длине 100):

| пул | 53 | 58 | 60 | 64 | 66 | 70 | всего |
|---|---|---|---|---|---|---|---|
| один выстрел по запечатанному пулу, без наблюдения (этапы 53–70) | 2 | 6 | 7 | 6 | 7 | 3 | 31 из 72 |
| с наблюдением, прежний обучатель (этап 72) | 8 | 9 | 7 | 7 | 7 | 3 | 41 из 72 |
| с наблюдением, этап 74 | 9 | 10 | 8 | 11 | 7 | 6 | **51 из 72** |

На этапе 74 точно (100 / 100 / 100 %) и с меткой «уверен» — 37 семейств; «уверен» и неверно — ни одного.

Все шесть пулов уже открыты: по ним обучатель дорабатывался, так что это не слепая проверка. Слепая —
последняя ячейка: **напишите правило сами**.

Обучатель работает на процессоре; видеокарта нужна только сетям-соперникам.
Короткая проверка, что всё запускается: `%env MX_QUICK=1` до первой ячейки; все шесть пулов: `%env MX_ALL=1`.

In [ ]:
import os, subprocess
ROOT = os.environ.get('MX_ROOT', '/content')
QUICK = bool(os.environ.get('MX_QUICK'))
if os.path.exists(ROOT + '/MX-Project'):
    subprocess.run(['git', '-C', ROOT + '/MX-Project', 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/kireikosasha/MX-Project', ROOT + '/MX-Project'], check=True)
M5 = ROOT + '/MX-Project/research_m5'
CODE = 'f5253458f80cec03b3456d509d16a5e774a82af6'  # the code (stages 74 and 76; the stage-74 flags behave as recorded)
subprocess.run(['git', '-C', ROOT + '/MX-Project', 'checkout', '-q', CODE, '--', 'research_m5/cells.py', 'research_m5/sealed_run.py'], check=True)
!pip -q install python-sat
WORKERS = os.cpu_count()
print('ядер:', WORKERS)

## Обучатель на открытых семействах

Код и флаги — те же, что в записанном прогоне этапа 74. Семейства идут параллельно, по одному на ядро. По
умолчанию — пул 70 (двенадцать семейств, на двух ядрах Colab около получаса–часа: большинство кончается
за минуту-две, несколько упираются в предел 30 минут); с `MX_ALL=1` — все 72. Рядом печатается записанное:
числа могут немного отличаться, потому что предел времени и порог памяти зависят от машины.

In [ ]:
import glob, os, re, subprocess, time
POOLS = ['53', '58', '60', '64', '66', '70'] if os.environ.get('MX_ALL') else ['70']
FLAGS = '--minutes 30 --workers 1 --seconds 3 --shorten --reserve 21 --explore --frugal --slack 16 --guards --build 12 --flags --cascades --keep --deadline --watch --warm --patience 30 --part 100000 --tour 24 --anchor 65 --far 100'
names = [(p, n) for p in POOLS for n in sorted(os.listdir(f'{M5}/sealed/stage{p}_families'))]
if QUICK:
    names = [('53', '05-garden'), ('64', '02-subway-turnstile'), ('58', '08-laundromat'), ('66', '04-hotel-reception')]
groups = [names[i::WORKERS] for i in range(WORKERS)]
os.makedirs('/content/run', exist_ok=True)
procs = []
for g, part in enumerate(groups):
    d = f'/content/run/g{g}'
    os.makedirs(d, exist_ok=True)
    for p, n in part:
        if not os.path.exists(f'{d}/{p}-{n}'):
            os.symlink(f'{M5}/sealed/stage{p}_families/{n}', f'{d}/{p}-{n}')
    procs.append(subprocess.Popen(f'cd {M5} && python sealed_run.py {d} {FLAGS} 2>/dev/null | grep "^[0-9][0-9]-" > /content/run/out{g}.txt', shell=True))
t0 = time.time()
for p in procs:
    p.wait()
print(f'{time.time() - t0:.0f} s')
mine = {}
for f in glob.glob('/content/run/out*.txt'):
    for line in open(f):
        mine[line.split(' ')[0]] = line.strip()
recorded = {line.split(' ')[0]: line.strip() for line in open(f'{M5}/results/stage74/results.txt')}
acc = lambda s: (re.search(r'100 / 500: [\d.]+ / ([\d.]+)', s) or [None, '?'])[1]
taken = 0
for p, n in names:
    k = f'{p}-{n}'
    line = mine.get(k, k + ': (нет)')
    taken += acc(line) != '?' and float(acc(line)) >= 95
    print(line)
    print('   записано на длине 100:', acc(recorded.get(k, '')), '%', '(не уверен)' if 'NOT SURE' in recorded.get(k, '') else '')
print(f'взято (95 % и выше на длине 100): {taken} из {len(names)}')

## Соперники на видеокарте

LSTM, трансформер и ALiBi на тех же семействах и по тому же протоколу наблюдателя (`--watch`): каждое
начало учебных рассказов — пример; учение на 300 рассказах и на в 40 раз большем числе, три зерна, 3000
шагов. Засчитывается лучшее. Записанное: на пуле 70 LSTM на 300 рассказах берёт 2 из 12, на 12 000 — 5 из 12; ALiBi на 12 000 (одно зерно) — 2, трансформер — 0; обучатель — 6 из 12 (`results/stage74/rivals_watch_pool70.txt`).

In [ ]:
for p in POOLS if not QUICK else []:
    fams = ' '.join(n for q, n in names if q == p)
    !cd {M5} && python cells_rivals.py --folder sealed/stage{p}_families {fams} --device cuda --watch --kinds lstm transformer alibi 2>/dev/null | grep "mean of"

## Автоматы реальных протоколов (этап 76)

Внешняя проверка: автоматы Мили, которые другие люди сняли с настоящих реализаций (коллекция AALpy,
`external/SOURCE.md`): устройства Bluetooth Low Energy, серверы TLS, брокеры MQTT, стеки TCP. Мерка — найдена ли
**в точности та же машина** (обход по парам состояний, `external/mealy_dot.py --same`).

Записано: замороженным кодом этапа 74 и 40 вопросами — 2 из 23 (неудача по названному заранее порогу); с чтением
ответа «с хода» (`--mealy --reach`) и бюджетом до 200 вопросов — 10 из 23 в точности, от 23 до 140 вопросов на
модель, и две машины «уверен», которые меньше настоящих (серверы NSS и JSSE). Ниже — девять устройств Bluetooth и
пять серверов TLS; модели MQTT и TCP на 15–57 состояний не сходятся и требуют много памяти.

In [ ]:
PFLAGS = FLAGS + ' --deep 24 --mealy --reach --asks 200 --save /content/found'
pnames = sorted(d for d in os.listdir(f'{M5}/external/protocols') if d[:2].isdigit())[:4 if QUICK else 14]
procs = []
for g in range(WORKERS):
    d = f'/content/proto/g{g}'
    os.makedirs(d, exist_ok=True)
    for n in pnames[g::WORKERS]:
        if not os.path.exists(f'{d}/{n}'):
            os.symlink(f'{M5}/external/protocols/{n}', f'{d}/{n}')
    procs.append(subprocess.Popen(f'cd {M5} && python sealed_run.py {d} {PFLAGS} 2>/dev/null | grep "^[0-9][0-9]-" > /content/proto/out{g}.txt', shell=True))
for p in procs:
    p.wait()
!cat /content/proto/out*.txt | sort | cut -c1-150
!cd {M5} && python external/mealy_dot.py --same /content/found/*.pkl

## Своё семейство — слепая проверка

Напишите правило сами — обучатель о нём ничего не знает. Нужны `ALPHABET` (2–8 слов) и `answer(seq)`,
которая для любого списка слов возвращает короткую строку. Ниже пример: лифт на пять этажей, который
нельзя сдвинуть, пока открыты двери. Папок с правилами может быть сколько угодно (`02-...`, `03-...`).

In [ ]:
os.makedirs('/content/mine/01-my-family', exist_ok=True)
open('/content/mine/01-my-family/family.py', 'w').write('''
ALPHABET = ["up", "down", "open", "close"]
def answer(seq):
    floor, doors = 0, False
    for t in seq:
        if t == "up" and not doors: floor = min(4, floor + 1)
        elif t == "down" and not doors: floor = max(0, floor - 1)
        elif t == "open": doors = True
        elif t == "close": doors = False
    return f"{floor} {'open' if doors else 'shut'}"
''')
!cd {M5} && python sealed_run.py /content/mine {FLAGS} 2>/dev/null